# Data Pipeline cho PhysioNet/CinC 2019 Sepsis

Notebook này tạo **3 fold dataframe** để dùng cho cross validation sau này. Theo `DATASET_OVERVIEW.md`, mỗi file `.psv` tương ứng với **1 bệnh nhân**, mỗi dòng là **1 giờ trong ICU**, và tên file được dùng để tạo `patient_id`.

Điểm quan trọng để tránh **data leakage**:

- Notebook trước hết lưu 3 fold trung gian đã được đọc file, thêm `patient_id`, và lọc outlier.
- Không fit median imputation trên toàn bộ dataset trước khi chia train/validation.
- Khi chạy từng vòng CV, median theo `(Age, Gender)` phải được fit từ **training folds của vòng đó**, rồi mới apply cho cả train và validation của vòng đó.

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR if (CURRENT_DIR / "cleaning.py").exists() else CURRENT_DIR / "data"
PROJECT_ROOT = DATA_DIR.parent
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 3
RANDOM_STATE = 42
WINDOW_SIZE = 6
DELTA_LAG_HOURS = 1
FEATURE_COLUMNS = ["HR", "O2Sat", "Temp", "SBP", "Resp"]
IMPUTE_COLUMNS = ["HR", "O2Sat", "Temp"]

def load_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

cleaning = load_module("cleaning", DATA_DIR / "cleaning.py")
feature_engineering = load_module("feature_engineering", DATA_DIR / "feature-engineering.py")

filter_absolute_outliers = cleaning.filter_absolute_outliers
fit_age_gender_normal_values = cleaning.fit_age_gender_normal_values
forward_fill_impute = cleaning.forward_fill_impute

print(f"DATA_DIR: {DATA_DIR}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

## 1. Tìm file `.psv` và tạo manifest bệnh nhân

Dataset local có thể nằm trực tiếp trong `data/raw/training_setA`, `data/raw/training_setB`, hoặc có thêm lớp thư mục con tùy cách giải nén. Vì vậy cell dưới dùng `rglob("*.psv")` để tìm toàn bộ file bệnh nhân.

`patient_id` được lấy từ tên file, ví dụ `p000001.psv` thành `p000001`.

In [ ]:
psv_files = sorted(RAW_DIR.rglob("*.psv"))
if not psv_files:
    raise FileNotFoundError(
        "Không tìm thấy file .psv trong data/raw. Hãy tải hoặc giải nén dataset vào data/raw trước."
    )

manifest = pd.DataFrame({"file_path": psv_files})
manifest["patient_id"] = manifest["file_path"].map(lambda p: p.stem)
manifest["source"] = manifest["file_path"].map(
    lambda p: "A" if "training_setA" in str(p) else ("B" if "training_setB" in str(p) else "unknown")
)

print(f"Số bệnh nhân tìm thấy: {len(manifest):,}")
display(manifest.head())
display(manifest["source"].value_counts(dropna=False).rename("n_patients"))

## 2. Chia 3 fold ở cấp bệnh nhân

Ta shuffle danh sách bệnh nhân với seed cố định rồi chia thành 3 fold. Việc chia theo bệnh nhân giúp tránh leakage giữa các timestep của cùng một bệnh nhân khi cross validation.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
shuffled_idx = rng.permutation(manifest.index.to_numpy())
fold_indices = np.array_split(shuffled_idx, N_FOLDS)

manifest["fold"] = -1
for fold_number, indices in enumerate(fold_indices, start=1):
    manifest.loc[indices, "fold"] = fold_number

fold_summary = manifest.groupby(["fold", "source"]).size().unstack(fill_value=0)
fold_summary["total"] = fold_summary.sum(axis=1)
display(fold_summary)

## 3. Hàm đọc một bệnh nhân và thêm `patient_id`

Mỗi `.psv` là một chuỗi thời gian của một bệnh nhân. Hàm dưới đọc file bằng separator `|`, thêm `patient_id`, `source`, và `source_file` để trace ngược về file gốc.

In [ ]:
def read_patient_psv(file_path: Path, source: str) -> pd.DataFrame:
    df = pd.read_csv(file_path, sep="|")
    df.insert(0, "patient_id", file_path.stem)
    df.insert(1, "source", source)
    df.insert(2, "source_file", file_path.name)
    return df

## 4. Cleaning bước đầu không leakage theo từng bệnh nhân

Ở bước tạo 3 fold trung gian, ta chỉ làm các xử lý không cần thống kê từ fold khác:

1. Đọc từng file bệnh nhân.
2. Lọc outlier tuyệt đối: `Temp`, `O2Sat`, `HR` ngoài range sinh lý sẽ thành `NaN`.
3. Giữ lại dữ liệu ở trạng thái `preimpute` để bước CV fit median đúng cách.

Chưa gọi `forward_fill_impute` và chưa tạo moving/delta/expanding ở đây, vì imputation phải được fit riêng trong từng vòng CV từ training folds.

In [ ]:
def process_patient_file_preimpute(file_path: Path, source: str) -> pd.DataFrame:
    df = read_patient_psv(file_path, source)
    df = filter_absolute_outliers(df)
    return df

## 5. Tạo 3 dataframe fold và lưu CSV

Cell dưới tạo lần lượt `fold_1_df`, `fold_2_df`, `fold_3_df`, sau đó lưu ra:

- `data/processed/fold_1_preimpute.csv`
- `data/processed/fold_2_preimpute.csv`
- `data/processed/fold_3_preimpute.csv`

Tên `_preimpute` nhắc rằng các file này chưa fit median imputation toàn cục. Đây là dữ liệu đầu vào sạch hơn cho bước cross validation.

In [ ]:
fold_dataframes = {}

for fold_number in range(1, N_FOLDS + 1):
    fold_manifest = manifest.loc[manifest["fold"] == fold_number].copy()
    patient_parts = []

    for row in fold_manifest.itertuples(index=False):
        patient_parts.append(process_patient_file_preimpute(row.file_path, row.source))

    fold_df = pd.concat(patient_parts, ignore_index=True)
    fold_dataframes[fold_number] = fold_df

    output_path = PROCESSED_DIR / f"fold_{fold_number}_preimpute.csv"
    fold_df.to_csv(output_path, index=False)
    print(
        f"Fold {fold_number}: {fold_manifest.shape[0]:,} bệnh nhân, "
        f"{fold_df.shape[0]:,} dòng, {fold_df.shape[1]:,} cột -> {output_path}"
    )

fold_1_df = fold_dataframes[1]
fold_2_df = fold_dataframes[2]
fold_3_df = fold_dataframes[3]

## 6. Fit median theo từng vòng CV để tránh data leakage

Khi một fold được dùng làm validation, median fallback phải được fit từ **2 fold còn lại**. Sau đó dùng cùng bộ median đó để impute cho cả train và validation, rồi mới tạo feature engineering.

Ví dụ:

- Vòng 1: fit median trên fold 2 + fold 3, validate trên fold 1.
- Vòng 2: fit median trên fold 1 + fold 3, validate trên fold 2.
- Vòng 3: fit median trên fold 1 + fold 2, validate trên fold 3.

In [ ]:
def ensure_patient_id_column(df: pd.DataFrame) -> pd.DataFrame:
    if "patient_id" in df.columns:
        return df
    if df.index.name == "patient_id" or "patient_id" in df.index.names:
        return df.reset_index()
    raise KeyError("Không tìm thấy cột patient_id. Hãy chạy lại các cell tạo fold preimpute từ đầu.")


def build_cv_split_with_fold_median_impute(
    fold_dataframes: dict[int, pd.DataFrame],
    valid_fold: int,
    impute_columns: list[str] = IMPUTE_COLUMNS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.concat(
        [df for fold_id, df in fold_dataframes.items() if fold_id != valid_fold],
        ignore_index=True,
    )
    valid_df = fold_dataframes[valid_fold].copy()
    train_df = ensure_patient_id_column(train_df)
    valid_df = ensure_patient_id_column(valid_df)

    fitted_values = fit_age_gender_normal_values(
        train_df,
        columns=impute_columns,
        patient_col="patient_id",
    )

    train_df = forward_fill_impute(
        train_df,
        columns=impute_columns,
        patient_col="patient_id",
        fitted_age_gender_values=fitted_values,
    )
    valid_df = forward_fill_impute(
        valid_df,
        columns=impute_columns,
        patient_col="patient_id",
        fitted_age_gender_values=fitted_values,
    )
    train_df = ensure_patient_id_column(train_df)
    valid_df = ensure_patient_id_column(valid_df)

    train_df = feature_engineering.add_all_feature_engineering(
        train_df,
        columns=FEATURE_COLUMNS,
        window_size=WINDOW_SIZE,
        delta_lag_hours=DELTA_LAG_HOURS,
        patient_col="patient_id",
    )
    valid_df = feature_engineering.add_all_feature_engineering(
        valid_df,
        columns=FEATURE_COLUMNS,
        window_size=WINDOW_SIZE,
        delta_lag_hours=DELTA_LAG_HOURS,
        patient_col="patient_id",
    )
    train_df = ensure_patient_id_column(train_df)
    valid_df = ensure_patient_id_column(valid_df)

    return train_df, valid_df


cv_splits = {}
for valid_fold in range(1, N_FOLDS + 1):
    train_df, valid_df = build_cv_split_with_fold_median_impute(fold_dataframes, valid_fold)
    cv_splits[valid_fold] = {"train": train_df, "valid": valid_df}
    print(
        f"CV vòng {valid_fold}: train={train_df['patient_id'].nunique():,} bệnh nhân, "
        f"valid={valid_df['patient_id'].nunique():,} bệnh nhân"
    )

## 7. Lưu optional các split CV đã impute và feature engineering đúng cách

Nếu muốn lưu trực tiếp dữ liệu đã impute và feature engineering cho từng vòng CV, chạy cell dưới. Mỗi vòng có một file train và một file validation riêng, vì median được fit khác nhau tùy validation fold.

Các file được lưu:

- `cv1_train.csv`, `cv1_valid.csv`
- `cv2_train.csv`, `cv2_valid.csv`
- `cv3_train.csv`, `cv3_valid.csv`

In [ ]:
SAVE_CV_SPLITS = True

if SAVE_CV_SPLITS:
    for valid_fold, split in cv_splits.items():
        train_path = PROCESSED_DIR / f"cv{valid_fold}_train.csv"
        valid_path = PROCESSED_DIR / f"cv{valid_fold}_valid.csv"
        split["train"].to_csv(train_path, index=False)
        split["valid"].to_csv(valid_path, index=False)
        print(f"Đã lưu CV vòng {valid_fold}: {train_path} và {valid_path}")

## 8. Kiểm tra nhanh output

Cell này xem kích thước từng fold, vài dòng đầu, và tỷ lệ label để chắc chắn file sau xử lý vẫn giữ được `SepsisLabel`.

In [ ]:
summary_rows = []
for fold_number, fold_df in fold_dataframes.items():
    row = {
        "fold": fold_number,
        "n_rows": len(fold_df),
        "n_patients": fold_df["patient_id"].nunique(),
        "n_columns": fold_df.shape[1],
    }
    if "SepsisLabel" in fold_df.columns:
        row["sepsis_label_mean"] = fold_df["SepsisLabel"].mean()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
display(fold_1_df.head())